<a href="https://colab.research.google.com/github/Balachandar-Ganesan/GraphRAG/blob/main/200_1_BuildYourOwnLLMJAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-JAX.txt

--2026-03-10 08:29:46--  https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-JAX.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 17892 (17K) [text/plain]
Saving to: ‘TinyStories-JAX.txt.1’

TinyStories-JAX.txt 100%[===================>]  17.47K  --.-KB/s    in 0s      

2026-03-10 08:29:46 (135 MB/s) - ‘TinyStories-JAX.txt.1’ saved [17892/17892]



In [ ]:
import jax
import jax.numpy as jnp
import optax
from flax import linen as nn
from flax.training import train_state
import tiktoken
import numpy as np

# Load and tokenize
with open("/content/TinyStories-JAX.txt", "r") as f:
    book_contents = f.read()

tokenizer = tiktoken.get_encoding("o200k_base")
sequence_len = 60

def get_data(data, n):
    tokens = tokenizer.encode(data,disallowed_special=())
    X = [tokens[i : n + i] for i in range(len(tokens) - n)]
    y = [tokens[i + 1 : n + i + 1] for i in range(len(tokens) - n)]
    return jnp.array(X), jnp.array(y)

X_data, y_data = get_data(book_contents, sequence_len)



In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn

class TinyTransformer(nn.Module):
    vocab_size: int
    embed_dim: int
    num_heads: int = 4

    @nn.compact
    def __call__(self, x):
        # 1. Embedding
        x = nn.Embed(num_embeddings=self.vocab_size, features=self.embed_dim)(x)

        # 2. Correct Causal Masking
        # Create a mask of shape (1, 1, seq_len, seq_len)
        # SelfAttention expects [batch, heads, q_len, kv_len]
        batch_size, seq_len, _ = x.shape
        mask = nn.make_causal_mask(jnp.ones((batch_size, seq_len)))

        # 3. Self-Attention
        # Ensure mask is passed as a keyword argument
        x = nn.SelfAttention(num_heads=self.num_heads)(x, mask=mask)

        # 4. MLP / Output
        x = nn.Dense(features=self.embed_dim * 2)(x)
        x = nn.relu(x)
        logits = nn.Dense(features=self.vocab_size)(x)
        return logits

# Updated Initialization
model = TinyTransformer(vocab_size=tokenizer.n_vocab, embed_dim=128)
# Use a real PRNGKey and a correctly shaped dummy input
key = jax.random.PRNGKey(0)
dummy_input = jnp.ones((1, sequence_len), dtype=jnp.int32)
variables = model.init(key, dummy_input)
params = variables['params']


In [ ]:
# Initialize State
tx = optax.adam(learning_rate=0.001)
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

@jax.jit
def train_step(state, batch_x, batch_y):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x)
        # Flatten for CrossEntropy
        loss = optax.softmax_cross_entropy_with_integer_labels(logits, batch_y).mean()
        return loss

    grad_fn = jax.value_and_grad(loss_fn)
    loss, grads = grad_fn(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

# Simple training loop
batch_size = 5
for epoch in range(20):
    for i in range(0, len(X_data), batch_size):
        batch_x = X_data[i:i+batch_size]
        batch_y = y_data[i:i+batch_size]
        state, loss = train_step(state, batch_x, batch_y)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


Epoch 1, Loss: 6.8764
Epoch 2, Loss: 4.9666
Epoch 3, Loss: 4.1912
Epoch 4, Loss: 3.8897
Epoch 5, Loss: 3.5026
Epoch 6, Loss: 2.9827
Epoch 7, Loss: 2.6620
Epoch 8, Loss: 2.3360
Epoch 9, Loss: 2.0709
Epoch 10, Loss: 1.8472
Epoch 11, Loss: 1.6082
Epoch 12, Loss: 1.2260
Epoch 13, Loss: 0.9336
Epoch 14, Loss: 0.9072
Epoch 15, Loss: 0.7837
Epoch 16, Loss: 0.6548
Epoch 17, Loss: 0.6887
Epoch 18, Loss: 0.6765
Epoch 19, Loss: 0.5393
Epoch 20, Loss: 0.5493


In [ ]:
def generate_jax(prompt, max_len=15):
    tokens = tokenizer.encode(prompt)
    for _ in range(max_len):
        input_tokens = jnp.array([tokens[-sequence_len:]])
        logits = state.apply_fn({'params': state.params}, input_tokens)
        next_token = jnp.argmax(logits[0, -1, :])
        tokens.append(int(next_token))
    print(tokenizer.decode(tokens))



generate_jax("three brothers")

three brothers.
A man gets stuck in a time loop and must prevent a political assassination
